# Brecha digital e inclusión tecnológica

Práctica Profesionalizante I
Barrera Ivan Gonzalo

En este notebook limpio la base que pasó la cátedra, miro ingreso, educación, región, computadora e internet, y armo los gráficos del informe.

La base tiene 1000 hogares. Sale del script de la cátedra, con semilla 42. Si se vuelve a generar, da el mismo archivo.


## 1. Carga

Leo el csv y miro el tamaño. Si lo abro desde la carpeta `notebooks`, el archivo está en `../data`. Si lo subo suelto a Colab, lo busca al lado del notebook.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

if Path("../data/eph_tecnologia_microdatos.csv").exists():
    RAIZ = Path("..")
elif Path("data/eph_tecnologia_microdatos.csv").exists():
    RAIZ = Path(".")
else:
    RAIZ = Path(".")

ruta = RAIZ / "data" / "eph_tecnologia_microdatos.csv"
if not ruta.exists():
    ruta = Path("eph_tecnologia_microdatos.csv")

df_raw = pd.read_csv(ruta)
print("filas, columnas:", df_raw.shape)
print(df_raw.head())
print()
print("nulos:")
print(df_raw.isna().sum())


## 2. Limpieza

En la clase 2 vimos los códigos de no respuesta de la EPH: -9, -1, 99 y 999. En este archivo no aparecen. Igual los paso a nulo, por si la cátedra cambia la base.

El 9 sí está, pero no significa lo mismo en todas las columnas. En nivel educativo, computadora e internet es "sin respuesta". En el decil, 9 es el noveno decil, y además hay un hogar con 9 integrantes. Si reemplazo el 9 en todo el dataframe, me como esos casos. Por eso lo hago solo en tres columnas.


In [ ]:
df = df_raw.copy()

df = df.replace([-9, -1, 99, 999], np.nan)

for col in ["NIVEL_ED", "IV12_3", "II7"]:
    df[col] = df[col].replace(9, np.nan)

print("nulos despues de limpiar:")
print(df.isna().sum())
print()
print("IX_TOT == 9 (integrantes, no es sin respuesta):", int((df_raw["IX_TOT"] == 9).sum()))
print("DECIFR == 9 (decil real):", int((df_raw["DECIFR"] == 9).sum()))


## 3. Etiquetas y archivo limpio

Paso los códigos a nombres para poder leer los gráficos. La guía nombra el ingreso per cápita, pero la base trae el ingreso total del hogar (ITF). Lo calculo dividiendo por la cantidad de integrantes.

No imputo. Si alguien no contestó si tiene PC, no le invento un sí o un no. Los porcentajes de más abajo salen de los que sí contestaron.


In [ ]:
regiones = {
    1: "GBA",
    40: "NOA",
    41: "NEA",
    42: "Cuyo",
    43: "Pampeana",
    44: "Patagonia",
}
niveles = {
    1: "Primaria incompleta",
    2: "Primaria completa",
    3: "Secundaria incompleta",
    4: "Secundaria completa",
    5: "Superior o universitaria completa",
}

df["region"] = df["REGION"].map(regiones)
df["nivel_ed"] = df["NIVEL_ED"].map(niveles)
df["tiene_pc"] = df["IV12_3"].map({1.0: 1, 2.0: 0})
df["tiene_internet"] = df["II7"].map({1.0: 1, 2.0: 0})
df["ipcf"] = df["ITF"] / df["IX_TOT"]
df["decil"] = df["DECIFR"].round().astype("Int64")

# si no fijo el orden, pandas los acomoda alfabetico y Cuyo queda primero
df["nivel_ed"] = pd.Categorical(df["nivel_ed"], categories=list(niveles.values()), ordered=True)
df["region"] = pd.Categorical(df["region"], categories=list(regiones.values()), ordered=True)

salida = RAIZ / "data" / "df_clean.csv"
if not (RAIZ / "data").exists():
    salida = Path("df_clean.csv")
df.to_csv(salida, index=False)
print("guarde", salida)


## 4. Descriptivas

Media, mediana, mínimo, máximo y desvío. En el ingreso miro también los percentiles, porque la media se va para arriba con unos pocos hogares de ingreso alto.


In [ ]:
print("Integrantes del hogar")
print(df["IX_TOT"].describe())
print()
print("Ingreso total familiar")
print(df["ITF"].describe())
print("percentil 25 y 75:", df["ITF"].quantile(0.25), df["ITF"].quantile(0.75))
print()
print("Ingreso por persona (ITF / integrantes)")
print(df["ipcf"].describe())
print()
print("PC: 1 = si, 0 = no")
print(df["tiene_pc"].value_counts(dropna=False))
print("porcentaje con PC, sobre los que contestaron:", round(df["tiene_pc"].mean() * 100, 1))
print()
print("Internet fija: 1 = si, 0 = no")
print(df["tiene_internet"].value_counts(dropna=False))
print("porcentaje con internet:", round(df["tiene_internet"].mean() * 100, 1))
print()
sin_ambos = ((df["tiene_pc"] == 0) & (df["tiene_internet"] == 0)).sum()
print("sin PC y sin internet:", int(sin_ambos))


### Gráficos univariados

El histograma del ingreso queda cargado a la izquierda. El boxplot muestra la cola de ingresos altos. El de educación es para ver cuántos hogares hay en cada nivel, nada más.


In [ ]:
figuras = RAIZ / "figuras"
figuras.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 4.4))
ax.hist(df["ITF"].dropna() / 1e6, bins=28, color="#4C78A8", edgecolor="white")
ax.set_title("Ingreso total familiar")
ax.set_xlabel("Millones de pesos")
ax.set_ylabel("Cantidad de hogares")
fig.tight_layout()
fig.savefig(figuras / "hist_ingreso.png", dpi=130)
plt.show()

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.boxplot(df["ITF"].dropna() / 1e6, vert=True, widths=0.45)
ax.set_title("Ingreso total familiar")
ax.set_ylabel("Millones de pesos")
ax.set_xticks([1])
ax.set_xticklabels(["Hogares"])
fig.tight_layout()
fig.savefig(figuras / "box_ingreso.png", dpi=130)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.4))
df["nivel_ed"].value_counts().sort_index().plot(kind="bar", ax=ax, color="#4C78A8")
ax.set_title("Hogares según nivel educativo del jefe")
ax.set_xlabel("")
ax.set_ylabel("Cantidad de hogares")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
fig.savefig(figuras / "barra_nivel.png", dpi=130)
plt.show()


## 5. Cruces

Tabla de contingencia y porcentaje de tenencia por nivel educativo y por decil. El porcentaje usa la media de 0/1, o sea que los que no contestaron no entran en la cuenta.


In [ ]:
print("Tabla de contingencia: nivel educativo x tiene PC")
print(pd.crosstab(df["nivel_ed"], df["tiene_pc"], dropna=False))
print()
print("Tabla de contingencia: decil x tiene internet")
print(pd.crosstab(df["decil"], df["tiene_internet"], dropna=False))
print()

def tasas(agrupador):
    base = df.dropna(subset=[agrupador])
    tabla = (
        base.groupby(agrupador, observed=True)[["tiene_pc", "tiene_internet"]]
        .mean()
        .mul(100)
        .round(1)
    )
    tabla.columns = ["% con PC", "% con internet"]
    hogares = base.groupby(agrupador, observed=True).size().rename("hogares")
    return pd.concat([hogares, tabla], axis=1)

print(tasas("nivel_ed"))
print()
print(tasas("decil"))

edu = tasas("nivel_ed")
deciles = tasas("decil")

figuras = RAIZ / "figuras"

fig, ax = plt.subplots(figsize=(9, 4.8))
edu[["% con PC", "% con internet"]].rename(columns={"% con PC": "PC", "% con internet": "Internet"}).plot(
    kind="bar", ax=ax, color=["#4C78A8", "#F58518"], width=0.78
)
ax.set_title("PC e internet según nivel educativo")
ax.set_ylabel("Porcentaje")
ax.set_xlabel("")
ax.set_ylim(0, 100)
ax.legend(loc="upper left")
ax.yaxis.grid(True, linestyle="--", alpha=0.35)
ax.set_axisbelow(True)
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
fig.savefig(figuras / "barra_educacion.png", dpi=130)
plt.show()

fig, ax = plt.subplots(figsize=(9, 4.8))
deciles[["% con PC", "% con internet"]].rename(columns={"% con PC": "PC", "% con internet": "Internet"}).plot(
    kind="bar", ax=ax, color=["#4C78A8", "#F58518"], width=0.78
)
ax.set_title("PC e internet según decil de ingreso")
ax.set_ylabel("Porcentaje")
ax.set_xlabel("Decil")
ax.set_ylim(0, 100)
ax.legend(loc="upper left")
ax.yaxis.grid(True, linestyle="--", alpha=0.35)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(figuras / "barra_decil.png", dpi=130)
plt.show()

print()
print("correlacion decil y PC:", round(df["decil"].astype(float).corr(df["tiene_pc"]), 3))
print("correlacion decil e internet:", round(df["decil"].astype(float).corr(df["tiene_internet"]), 3))
print("correlacion nivel educativo y PC:", round(df["NIVEL_ED"].corr(df["tiene_pc"]), 3))
print("correlacion nivel educativo e internet:", round(df["NIVEL_ED"].corr(df["tiene_internet"]), 3))
print("correlacion ITF y PC:", round(df["ITF"].corr(df["tiene_pc"]), 3))
print("correlacion ingreso por persona y PC:", round(df["ipcf"].corr(df["tiene_pc"]), 3))


## 6. Región

La consigna pide mirar la región. Yo esperaba que NOA y NEA quedaran bastante abajo de GBA. En esta base la diferencia es chica. Dejo la escala de 0 a 100, igual que en el gráfico de educación, para que se pueda comparar.

Si el organismo pidiera solo el GBA, no rehago la limpieza. Filtro y vuelvo a calcular las tablas:

`df_gba = df[df["REGION"] == 1]`


In [ ]:
print(tasas("region"))

reg = tasas("region")
figuras = RAIZ / "figuras"
fig, ax = plt.subplots(figsize=(9, 4.8))
reg[["% con PC", "% con internet"]].rename(columns={"% con PC": "PC", "% con internet": "Internet"}).plot(
    kind="bar", ax=ax, color=["#4C78A8", "#F58518"], width=0.78
)
ax.set_title("PC e internet según región")
ax.set_ylabel("Porcentaje")
ax.set_xlabel("")
ax.set_ylim(0, 100)
ax.legend(loc="lower right")
ax.yaxis.grid(True, linestyle="--", alpha=0.35)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(figuras / "barra_region.png", dpi=130)
plt.show()

# chequeo rapido del filtro, por si piden solo GBA
df_gba = df[df["REGION"] == 1]
print("hogares GBA:", len(df_gba))
print("PC en GBA:", round(df_gba["tiene_pc"].mean() * 100, 1))
print("internet en GBA:", round(df_gba["tiene_internet"].mean() * 100, 1))


## 7. Regresión lineal

Ajusto una recta simple: tener PC (0 o 1) según el decil. No es el modelo ideal para un sí/no, pero es la regresión que vimos en clase. Sirve para chequear si el decil empuja para arriba, no para predecir un hogar puntual.

El nivel educativo no lo meto en la recta. Los códigos 1 a 5 no son una escala pareja (pasar de primaria incompleta a completa no es "lo mismo" que pasar de secundaria completa a universidad). Para educación me quedo con la tabla.


In [ ]:
reg = df.dropna(subset=["decil", "tiene_pc"]).copy()
x = reg["decil"].to_numpy(dtype=float)
y = reg["tiene_pc"].to_numpy(dtype=float)

b, a = np.polyfit(x, y, 1)
y_hat = a + b * x
r2 = 1 - ((y - y_hat) ** 2).sum() / ((y - y.mean()) ** 2).sum()

print("n usados:", len(reg))
print(f"tiene_pc = {a:.3f} + {b:.3f} * decil")
print(f"R2 = {r2:.3f}")
print(f"el modelo en el decil 1 estima {100 * (a + b * 1):.1f}%")
print(f"el modelo en el decil 10 estima {100 * (a + b * 10):.1f}%")


## Para el informe

Las tres hipótesis se ven en las tablas: a menor ingreso y menor educación, menos PC y menos internet fija. La región, en estos 1000 hogares, casi no se mueve.

Los porcentajes del informe salen de estas celdas. No los redondeé a mano en otro lado.
